In [2]:
### 프로젝트 루트 경로 설정 및 YAML 설정 파일 로드
# 주피터 노트북이 `notebook/` 폴더 안에서 실행될 때 발생하는 경로 깨짐 문제를 방지하고, 프로젝트 전역 설정(`config/default.yaml`)을 불러옵니다.
import os
import sys
import yaml
import json

# 주피터 노트북 실행 위치에 따른 프로젝트 루트 경로 자동 보정
current_dir = os.getcwd()
if current_dir.endswith("notebook"):
    os.chdir("..")

if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

def load_questions_by_id(path="data/evaluation/questions.jsonl"):
    questions = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            item = json.loads(line)
            questions[item["question_id"]] = item
    return questions

QUESTIONS_BY_ID = load_questions_by_id()

# 시스템 전역 설정 파일 로드
with open("config/default.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

print("설정 로드 완료:", config["generation"])

설정 로드 완료: {'provider': 'openai', 'model': 'gpt-5-nano', 'temperature': 0.1, 'top_p': 0.95, 'max_tokens': 4500, 'reasoning_effort': 'low', 'history': {'max_turns': 3}}


In [ ]:
### 연동 테스트용 Mock 청크 데이터 로드
# 사전에 정의한 가상 제안요청서 청크 데이터(`sample_chunks.jsonl`)를 `SearchResult` 객체로 파싱합니다.
import json
from src.retriever import SearchResult

mock_results = []
file_path = "samples/processed/sample_chunks.jsonl"

# 가상 청크 JSONL 파일을 읽어 SearchResult 객체 리스트로 변환
with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)
        result = SearchResult(
            chunk_id=data["chunk_id"],
            doc_id=data["doc_id"],
            text=data["text"],
            score=0.85,  # 가상 테스트용 임시 유사도 점수
            metadata=data["metadata"]
        )
        mock_results.append(result)

print(f"총 {len(mock_results)}개의 청크 데이터를 성공적으로 불러왔습니다!")

In [3]:
### Test Case 1: 문서 내 다중 조건 복합 질문 테스트
# 여러 청크에 나뉘어 있는 정보(주민등록번호 제외 조건, 1년 보관 기간)를 하나의 답변으로 정확히 종합하는지 검증합니다.
from src.rag_engine import generate_answer
question_1 = "기존 시스템에서 사용자 정보 이전할 때 제외해야 하는 항목이 뭐야? 그리고 개인정보 다운로드 기록은 얼마나 보관해야 해?"
response_1 = generate_answer(question_1, mock_results, config)

print(f"Q: {question_1}\n")
print(f"A:\n{response_1['answer']}")

Q: 기존 시스템에서 사용자 정보 이전할 때 제외해야 하는 항목이 뭐야? 그리고 개인정보 다운로드 기록은 얼마나 보관해야 해?

A:
- 제외 항목: 주민등록번호는 이전 대상에서 제외한다. [1]
- 개인정보 다운로드 기록 보관 기간: 개인정보 조회와 다운로드 기록을 1년간 보관해야 한다. [2]

참고 문서: sample_rfp.txt
Q: 기존 시스템에서 사용자 정보 이전할 때 제외해야 하는 항목이 뭐야? 그리고 개인정보 다운로드 기록은 얼마나 보관해야 해?

A:
- 기존 시스템에서 사용자 정보를 이전할 때 제외해야 하는 항목: 주민등록번호입니다. [1]

- 개인정보 다운로드 기록 보관 기간: 1년간 보관해야 합니다. [2]

참고 문서
- [1] 2026년 공공 AI 학습지원 플랫폼 구축 사업 (가상디지털진흥원) sample_rfp.txt
- [2] 2026년 공공 AI 학습지원 플랫폼 구축 사업 (가상디지털진흥원) sample_rfp.txt


In [4]:
### Test Case 2: 문서 외 질문에 대한 할루시네이션 방어 테스트
# 제공된 문서에 존재하지 않는 기능에 대해 아는 척 지어내지 않고, 단호하게 "확인할 수 없다"고 차단하는지 검증합니다.
question_2 = "이 플랫폼에 블록체인 연계 기능이 포함되어 있어?"
response_2 = generate_answer(question_2, mock_results, config)

print(f"Q: {question_2}\n")
print(f"A:\n{response_2['answer']}")

Q: 이 플랫폼에 블록체인 연계 기능이 포함되어 있어?

A:
- 제공된 문서에서 근거를 찾을 수 없어 확인할 수 없습니다. [1][2][3]

참고: sample_rfp.txt
Q: 이 플랫폼에 블록체인 연계 기능이 포함되어 있어?

A:
- 제공된 문서에 블록체인 연계 기능의 포함 여부에 대해 명시적으로 다루고 있지 않으므로, 현재 맥락으로는 확인할 수 없습니다. [1][2][3]

참고 문서: sample_rfp.txt (2026년 공공 AI 학습지원 플랫폼 구축 사업)


In [5]:
### Test Case 3: 제안 공고 조건 및 제출 방식 확인 테스트
# RFP의 마감일시와 제한 조건(이메일 제출 불가 등)을 정확하게 추출하여 안내하는지 검증합니다.
question_3 = "제안서 제출 마감일시는 언제이며, 이메일 제출이 가능한가요?"
response_3 = generate_answer(question_3, mock_results, config)

print(f"Q: {question_3}\n")
print(f"A:\n{response_3['answer']}")

Q: 제안서 제출 마감일시는 언제이며, 이메일 제출이 가능한가요?

A:
- 제안서 제출 마감일시는 2026년 8월 14일 17시입니다. [2]
- 이메일 제출은 인정되지 않습니다. 또한 제출은 나라장터를 통한 온라인 제출이며, 전자파일은 PDF 형식으로 등록해야 합니다. [2]

참고 문서: sample_rfp.txt (2026년 공공 AI 학습지원 플랫폼 구축 사업)


In [6]:
### Test Case 4: 시스템 요구사항 및 제약 조건 추출 테스트
# 문서 내에 명시된 기능적 필수 조건(출처 표시, 근거 없을 시 응답 제한)을 누락 없이 요약하는지 검증합니다.
question_4 = "AI 학습 도우미 기능에서 답변을 제공할 때 지켜야 할 필수 조건은 뭐야?"
response_4 = generate_answer(question_4, mock_results, config)

print(f"Q: {question_4}\n")
print(f"A:\n{response_4['answer']}")

Q: AI 학습 도우미 기능에서 답변을 제공할 때 지켜야 할 필수 조건은 뭐야?

A:
다음은 AI 학습 도우미가 답변을 제공할 때 지켜야 할 필수 조건입니다.

- 사용한 교육자료의 출처 표시: 도우미의 답변에 사용된 교육자료의 제목과 위치를 함께 표시해야 합니다. [1]

- 근거 기반 응답: 검색된 교육자료에 근거한 내용만 전달하고, 자료에 없는 내용을 사실처럼 생성해서는 안 됩니다. [2]

- 불확실성 처리: 근거를 찾지 못한 경우에는 “확인할 수 없다고 답변해야 한다.” [2]

참고한 문서:
- [1] 2026년 공공 AI 학습지원 플랫폼 구축 사업
- [2] 2026년 공공 AI 학습지원 플랫폼 구축 사업
Q: AI 학습 도우미 기능에서 답변을 제공할 때 지켜야 할 필수 조건은 뭐야?

A:
- AI 학습 도우미는 답변에 사용한 교육자료의 제목과 위치를 함께 표시해야 한다. [1]
- AI 학습 도우미는 검색된 교육자료에 없는 내용을 사실처럼 생성해서는 안 되며 근거를 찾지 못하면 확인할 수 없다고 답변해야 한다. [2]

참고한 문서: sample_rfp.txt


In [7]:
### Test Case 5: 문서에 없는 부가 정책 질문에 대한 방어 테스트
# 예산이나 보안 외의 사내 복리후생 성격의 질문에 대해 추측하지 않고 근거 부재를 올바르게 안내하는지 검증합니다.
question_5 = "이 프로젝트에 투입되는 개발자들의 식대 지원 기준은 어떻게 돼?"
response_5 = generate_answer(question_5, mock_results, config)

print(f"Q: {question_5}\n")
print(f"A:\n{response_5['answer']}")

Q: 이 프로젝트에 투입되는 개발자들의 식대 지원 기준은 어떻게 돼?

A:
- 제시된 RFP 문서에는 개발자 식대 지원 기준에 대한 구체적 내용이 명시되어 있지 않습니다. 따라서 이 질문에 대한 근거를 문서에서 확인할 수 없어 확인이 불가합니다. [1][2][3]

참고한 문서
- sample_rfp.txt (가상 제안요청서 1~3 편)
Q: 이 프로젝트에 투입되는 개발자들의 식대 지원 기준은 어떻게 돼?

A:
- 요청하신 개발자 식대 지원 기준에 관해, 제공된 문서에서 근거를 찾을 수 없어 확인할 수 없습니다 [1][2][3].

참고한 문서
- sample_rfp.txt (2026년 공공 AI 학습지원 플랫폼 구축 사업)


In [8]:
from src.rag_engine import build_context, SYSTEM_RULE
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model="gpt-5-nano", max_tokens=1000)
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_RULE + "\n\nContext:\n{context}"),
    ("human", "{question}")
])
chain = prompt | llm

msg = chain.invoke({
    "context": build_context(mock_results),
    "question": "이 플랫폼에 블록체인 연계 기능이 포함되어 있어?"
})
print(msg.response_metadata)
print(repr(msg.content))

{'token_usage': {'completion_tokens': 1000, 'prompt_tokens': 1711, 'total_tokens': 2711, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 1000, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E3eewYzfDiAGawIjiIdqi6gdpFpOO', 'service_tier': 'default', 'finish_reason': 'length', 'logprobs': None}
''


In [9]:
### End-to-End 연결 검증: profile별 동작 확인
from api_main import run

for profile in ["baseline", "openai"]:
    print(f"=== profile: {profile} ===")
    response = run("사업 예산과 수행 기간은 어떻게 돼?", profile=profile)
    print(response["answer"])
    print(f"(sources: {len(response['sources'])}건)\n")

=== profile: baseline ===
- 제공된 문서들에는 사업 예산(예산 규모)과 수행 기간의 구체적인 수치가 명시되어 있지 않습니다. 따라서 현재 근거만으로는 예산과 기간을 확인할 수 없습니다. [2][4][5]

참고한 문서 출처
- [2] 인천일자리플랫폼 정보시스템 구축 ISP 수립용역.hwp
- [4] 전문대학 혁신지원사업 서영대학교 차세대 교육혁신지원시스템 3단계 구축 용역 재공고
- [5] 차세대 포털·학사 정보시스템 구축사업.pdf
(sources: 5건)

=== profile: openai ===
- 수행 기간: 서민금융 채팅 상담시스템 구축의 사업기간은 2024년 6월부터 2024년 9월까지로 총 4개월입니다. [1]
- 예산: 제공된 문서에서 사업 예산 금액은 명시되어 있지 않아 확인할 수 없습니다. [1]

참고한 문서: 서민금융 채팅 상담시스템 구축.hwp [1]
(sources: 5건)

(sources: 0건)

=== profile: baseline ===
- 제공된 문서에서 사업 예산과 수행 기간에 대한 구체적 수치를 확인할 수 없습니다. [1][2][3]

참고한 문서:
- 한국연구재단_2024년 기초학문자료센터 시스템 운영 및 연구성과물 DB구축 사업
- 인천광역시_인천일자리플랫폼 정보시스템 구축 ISP 수립용역.hwp
- 국방과학연구소_기록관리시스템 통합 활용 및 보안 환경 구축.hwp
(sources: 3건)

=== profile: openai ===
관련 문서 내용을 찾지 못했습니다. 원본 문서나 검색 조건을 다시 확인해 주세요.
(sources: 0건)

(sources: 0건)



In [10]:
# --- 셀 1: 정상 검색 → 답변+출처 확인 ---
from api_main import run, print_result
 
question = "한영대학교 특성화 맞춤형 교육환경 구축 사업의 예산과 수행 기간은 어떻게 돼?"
response = run(question, profile="baseline")
print_result(question, "baseline", response)

[질문] 한영대학교 특성화 맞춤형 교육환경 구축 사업의 예산과 수행 기간은 어떻게 돼?
[Profile] baseline

[답변]
- 예산: 130,000,000원 범위 내(VAT 포함) [2]
- 수행 기간: 계약일로부터 3개월(안정화기간 1개월 포함) [2]

참고 문서: 한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp [2]

[출처 5건]
1. 한국산업단지공단_산단 안전정보시스템 1차 구축 용역.hwp | 기관: 한국산업단지공단 | chunk_id: doc_095_chunk_0003 | score: 0.1592
2. 한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp | 기관: 한영대학 | chunk_id: doc_001_chunk_0001 | score: 0.1585
3. 한국산업단지공단_산단 안전정보시스템 1차 구축 용역.hwp | 기관: 한국산업단지공단 | chunk_id: doc_095_chunk_0007 | score: 0.1558
4. 그랜드코리아레저(주)_2024년도 GKL  그룹웨어 시스템 구축 용역.hwp | 기관: 그랜드코리아레저(주) | chunk_id: doc_094_chunk_0010 | score: 0.1456
5. 사단법인 보험개발원_실손보험 청구 전산화 시스템 구축 사업.hwp | 기관: 사단법인 보험개발원 | chunk_id: doc_084_chunk_0032 | score: 0.1411


In [11]:
# --- 셀 2: 0건 검색 케이스 확인 ---
question_no_match = "존재하지 않는 사업에 대해 알려줘"
response_no_match = run(
    question_no_match,
    profile="baseline",
    filters={"agency": "존재하지-않는-기관"},
)
print_result(question_no_match, "baseline", response_no_match)

[질문] 존재하지 않는 사업에 대해 알려줘
[Profile] baseline

[답변]
관련 문서 내용을 찾지 못했습니다. 원본 문서나 검색 조건을 다시 확인해 주세요.

[출처 0건]


In [12]:
# --- 이슈 #29: 대화 히스토리·후속 질문 처리 데모 (3턴) ---
import yaml
from pathlib import Path
from src.parser_chunker import load_chunks_jsonl, demo_chunks
from src.retriever_factory import create_retriever
from src.rag_engine import generate_answer, condense_question

with open("config/default.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

chunks_path = Path(config["paths"]["chunks"])
chunks = load_chunks_jsonl(chunks_path) if chunks_path.exists() else demo_chunks()
retriever = create_retriever(chunks, config["retrieval"], "baseline")

history = []

def ask(question, history):
    search_question = condense_question(question, history, config)
    results = retriever.search(search_question, top_k=config["retrieval"]["top_k"], filters=None)
    response = generate_answer(question, results, config, history=history)
    print(f"[질문] {question}")
    if search_question != question:
        print(f"[검색용 재구성 질문] {search_question}")
    print(f"[답변]\n{response['answer']}\n")
    print(f"[출처 {len(response['sources'])}건]")
    for idx, src in enumerate(response["sources"], start=1):
        meta = src.get("metadata", {})
        print(f"{idx}. {meta.get('file_name', '출처 없음')} | 기관: {meta.get('agency', '기관 정보 없음')}")
    print("-" * 60)
    history.append({"question": question, "answer": response["answer"]})

ask("한영대학교 특성화 맞춤형 교육환경 구축 사업의 예산은 얼마야?", history)
ask("그럼 수행 기간은 어떻게 돼?", history)
ask("고려대학교 차세대 포털·학사 정보시스템 구축사업의 예산은 얼마야?", history)

[질문] 한영대학교 특성화 맞춤형 교육환경 구축 사업의 예산은 얼마야?
[답변]
- 예산은 130,000,000원(VAT 포함) 범위 내입니다. [2]

참고한 문서: 한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp

[출처 3건]
1. 한국산업단지공단_산단 안전정보시스템 1차 구축 용역.hwp | 기관: 한국산업단지공단
2. 한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp | 기관: 한영대학
3. 한국산업단지공단_산단 안전정보시스템 1차 구축 용역.hwp | 기관: 한국산업단지공단
------------------------------------------------------------
[질문] 그럼 수행 기간은 어떻게 돼?
[검색용 재구성 질문] 한영대학교 특성화 맞춤형 교육환경 구축 사업의 수행 기간은 얼마나 되며, 시작일과 종료일은 언제인가요?
[답변]
- 수행 기간은 계약일로부터 3개월이며, 안정화기간 1개월이 포함됩니다. [2]

참고한 문서: 한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp

[출처 3건]
1. 한국산업단지공단_산단 안전정보시스템 1차 구축 용역.hwp | 기관: 한국산업단지공단
2. 한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp | 기관: 한영대학
3. 한국산업단지공단_산단 안전정보시스템 1차 구축 용역.hwp | 기관: 한국산업단지공단
------------------------------------------------------------
[질문] 고려대학교 차세대 포털·학사 정보시스템 구축사업의 예산은 얼마야?
[답변]
- 제공된 문서에서 고려대학교 차세대 포털·학사 정보시스템 구축사업의 예산에 대한 구체적 금액은 확인할 수 없습니다. [2]
- 예산 항목의 존재 여부 및 세부 금액은 문서의 사업 범위 및 제안서 구성에서 명시되지 않았습니다. [3]

참고한 문서: 고려대학교_차세대 포털·학사 정보시스템 구

In [13]:
import time
from api_main import load_config, load_chunks
from src.retriever_factory import create_retriever
from src.rag_engine import build_context, build_llm, condense_question, SYSTEM_RULE
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

def diagnose(question: str, profile: str = "openai", config_path: str = "config/default.yaml", n_repeat: int = 3):
    config = load_config(config_path)
    chunks = load_chunks(config["paths"]["chunks"])
    retriever = create_retriever(chunks, config["retrieval"], profile)

    search_question = condense_question(question, None, config)
    results = retriever.search(search_question, top_k=config["retrieval"]["top_k"])
    context = build_context(results)

    llm = build_llm(config)  # StrOutputParser 없이 raw ChatOpenAI
    prompt = ChatPromptTemplate.from_messages([
        ("system", SYSTEM_RULE + "\n\nContext:\n{context}"),
        MessagesPlaceholder("history", optional=True),
        ("human", "{question}")
    ])
    chain = prompt | llm  # StrOutputParser 제거 -> AIMessage 그대로 받음

    for i in range(n_repeat):
        start = time.perf_counter()
        ai_msg = chain.invoke({"context": context, "question": question, "history": []})
        elapsed = time.perf_counter() - start

        meta = ai_msg.response_metadata
        usage = meta.get("token_usage", {})
        reasoning_tokens = usage.get("completion_tokens_details", {}).get("reasoning_tokens")

        print(f"--- 시도 {i+1} ({elapsed:.2f}s) ---")
        print(f"finish_reason: {meta.get('finish_reason')}")
        print(f"prompt_tokens: {usage.get('prompt_tokens')}, "
              f"completion_tokens: {usage.get('completion_tokens')}, "
              f"reasoning_tokens: {reasoning_tokens}")
        print(f"답변 길이: {len(ai_msg.content)}자")
        print(f"답변 미리보기: {ai_msg.content[:200]!r}")
        print(f"[전체 response_metadata] {meta}")
        print()

diagnose(
    "서울시립대학교의 '대입전형 자료', '학적 정보', '학생 활동 정보' 데이터는 각각 어느 부서에서 담당하여 수집 및 관리하고 있나요?",
    profile="openai",
    n_repeat=3,
)

--- 시도 1 (8.09s) ---
finish_reason: stop
prompt_tokens: 3356, completion_tokens: 628, reasoning_tokens: 512
답변 길이: 165자
답변 미리보기: '- 대입전형 자료(입학정보) 수집·관리: 입학처가 담당합니다. [4]\n- 학적 정보 현황 수집·관리: 교무처가 담당합니다. [4]\n- 학생 활동 정보 현황 수집·관리: 교무처가 담당합니다. [4]\n\n참고한 문서: 서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf [4]'
[전체 response_metadata] {'token_usage': {'completion_tokens': 628, 'prompt_tokens': 3356, 'total_tokens': 3984, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 512, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E6WcbriiItGffoHmFEI2AM8ClvQTs', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}

--- 시도 2 (3.94s) ---
finish_reason: stop
prompt_tokens: 3356, completion_tokens: 606, reasoning_tokens: 448
답변 길이: 259자
답변 미리보기: '- 대입전형 자

In [14]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
import time, yaml
from copy import deepcopy
from langchain_community.callbacks import get_openai_callback
from api_main import run, load_config

BASE_CONFIG_PATH = "config/default.yaml"

CONFIGS = {
    "A_nano_3000": {"model": "gpt-5-nano", "max_tokens": 3000},
    "B_nano_1500": {"model": "gpt-5-nano", "max_tokens": 1500},
    "C_mini_3000":  {"model": "gpt-5-mini", "max_tokens": 3000},
    "D_nano_4500":  {"model": "gpt-5-nano", "max_tokens": 4500},  # reasoning 토큰 소진 대응 상향안
}

QUESTIONS = [
    "한영대학교 특성화 맞춤형 교육환경 구축 사업의 예산과 수행 기간은 어떻게 돼?",
    "차세대 학사 정보시스템 구축에 대해 알려줘",
]

def make_temp_config(overrides: dict) -> str:
    cfg = deepcopy(load_config(BASE_CONFIG_PATH))
    cfg["generation"].update(overrides)
    tmp_path = f"config/_tmp_{overrides['model']}_{overrides['max_tokens']}.yaml"
    with open(tmp_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(cfg, f, allow_unicode=True)
    return tmp_path

results = []
for cfg_name, overrides in CONFIGS.items():
    tmp_path = make_temp_config(overrides)
    for q in QUESTIONS:
        start = time.perf_counter()
        with get_openai_callback() as cb:
            response = run(q, config_path=tmp_path, profile="openai")
        elapsed = time.perf_counter() - start
        cost = cb.total_cost
        results.append({
            "config": cfg_name, "model": overrides["model"], "max_tokens": overrides["max_tokens"],
            "question": q, "latency_sec": round(elapsed, 2),
            "prompt_tokens": cb.prompt_tokens, "prompt_tokens_cached": cb.prompt_tokens_cached, "completion_tokens": cb.completion_tokens,
            "answer_len": len(response["answer"]),
            "cost_usd": round(cost, 5), "answer": response["answer"],
        })

import pandas as pd
df = pd.DataFrame(results)
df

,config,model,max_tokens,question,latency_sec,prompt_tokens,prompt_tokens_cached,completion_tokens,answer_len,cost_usd,answer
0,A_nano_3000,gpt-5-nano,3000,한영대학교 특성화 맞춤형 교육환경 구축 사업의 예산과 수행 기간은 어떻게 돼?,2.34,3459,0,151,124,0.00023,"- 예산: 130,000,000원(VAT 포함) [1]\n- 수행 기간: 계약일로부..."
1,A_nano_3000,gpt-5-nano,3000,차세대 학사 정보시스템 구축에 대해 알려줘,10.30,3235,0,1141,1276,0.00062,차세대 학사 정보시스템 구축에 대해 요약하면 아래와 같습니다.\n\n- 목적 및 기...
2,B_nano_1500,gpt-5-nano,1500,한영대학교 특성화 맞춤형 교육환경 구축 사업의 예산과 수행 기간은 어떻게 돼?,2.47,3459,3328,153,126,0.00008,"- 예산: 130,000,000원(VAT 포함) [1]\n- 사업기간: 계약일로부터..."
3,B_nano_1500,gpt-5-nano,1500,차세대 학사 정보시스템 구축에 대해 알려줘,7.18,3235,3072,1033,1338,0.00044,다음은 제공된 문서들을 토대로 정리한 차세대 학사 정보시스템 구축 현황 및 주요 내...
4,C_mini_3000,gpt-5-mini,3000,한영대학교 특성화 맞춤형 교육환경 구축 사업의 예산과 수행 기간은 어떻게 돼?,3.26,3459,0,235,156,0.00133,"- 사업예산: 130,000,000원(부가가치세 포함)으로 책정되어 있음 [1]. ..."
5,C_mini_3000,gpt-5-mini,3000,차세대 학사 정보시스템 구축에 대해 알려줘,8.97,3235,0,1020,1262,0.00285,요청하신 “차세대 학사 정보시스템 구축”에 대해 제공된 문서들을 근거로 핵심 내용만...
6,D_nano_4500,gpt-5-nano,4500,한영대학교 특성화 맞춤형 교육환경 구축 사업의 예산과 수행 기간은 어떻게 돼?,2.53,3459,3328,155,130,0.00009,"- 예산: 130,000,000원 범위 내 (부가가치세 포함) [1]\n- 수행 기..."
7,D_nano_4500,gpt-5-nano,4500,차세대 학사 정보시스템 구축에 대해 알려줘,8.27,3235,3072,1181,1152,0.00050,다음은 차세대 학사 정보시스템 구축에 대해 맥락 문서의 내용을 바탕으로 정리한 요약...


In [15]:
NEW_CONFIGS = {
    "E_nano_8000": {"model": "gpt-5-nano", "max_tokens": 8000},   # nano, 예산 훨씬 더 키움
    "F_mini_1500": {"model": "gpt-5-mini", "max_tokens": 1500},   # mini, 예산 줄여봄
}

new_results = []
for cfg_name, overrides in NEW_CONFIGS.items():
    tmp_path = make_temp_config(overrides)
    for q in QUESTIONS:
        start = time.perf_counter()
        with get_openai_callback() as cb:
            response = run(q, config_path=tmp_path, profile="openai")
        elapsed = time.perf_counter() - start
        cost = cb.total_cost
        new_results.append({
            "config": cfg_name, "model": overrides["model"], "max_tokens": overrides["max_tokens"],
            "question": q, "latency_sec": round(elapsed, 2),
            "prompt_tokens": cb.prompt_tokens, "prompt_tokens_cached": cb.prompt_tokens_cached, "completion_tokens": cb.completion_tokens,
            "answer_len": len(response["answer"]),
            "cost_usd": round(cost, 5), "answer": response["answer"],
        })

df2 = pd.DataFrame(new_results)
df_all = pd.concat([df, df2], ignore_index=True)
df_all

,config,model,max_tokens,question,latency_sec,prompt_tokens,prompt_tokens_cached,completion_tokens,answer_len,cost_usd,answer
0,A_nano_3000,gpt-5-nano,3000,한영대학교 특성화 맞춤형 교육환경 구축 사업의 예산과 수행 기간은 어떻게 돼?,2.34,3459,0,151,124,0.00023,"- 예산: 130,000,000원(VAT 포함) [1]\n- 수행 기간: 계약일로부..."
1,A_nano_3000,gpt-5-nano,3000,차세대 학사 정보시스템 구축에 대해 알려줘,10.30,3235,0,1141,1276,0.00062,차세대 학사 정보시스템 구축에 대해 요약하면 아래와 같습니다.\n\n- 목적 및 기...
2,B_nano_1500,gpt-5-nano,1500,한영대학교 특성화 맞춤형 교육환경 구축 사업의 예산과 수행 기간은 어떻게 돼?,2.47,3459,3328,153,126,0.00008,"- 예산: 130,000,000원(VAT 포함) [1]\n- 사업기간: 계약일로부터..."
3,B_nano_1500,gpt-5-nano,1500,차세대 학사 정보시스템 구축에 대해 알려줘,7.18,3235,3072,1033,1338,0.00044,다음은 제공된 문서들을 토대로 정리한 차세대 학사 정보시스템 구축 현황 및 주요 내...
4,C_mini_3000,gpt-5-mini,3000,한영대학교 특성화 맞춤형 교육환경 구축 사업의 예산과 수행 기간은 어떻게 돼?,3.26,3459,0,235,156,0.00133,"- 사업예산: 130,000,000원(부가가치세 포함)으로 책정되어 있음 [1]. ..."
5,C_mini_3000,gpt-5-mini,3000,차세대 학사 정보시스템 구축에 대해 알려줘,8.97,3235,0,1020,1262,0.00285,요청하신 “차세대 학사 정보시스템 구축”에 대해 제공된 문서들을 근거로 핵심 내용만...
6,D_nano_4500,gpt-5-nano,4500,한영대학교 특성화 맞춤형 교육환경 구축 사업의 예산과 수행 기간은 어떻게 돼?,2.53,3459,3328,155,130,0.00009,"- 예산: 130,000,000원 범위 내 (부가가치세 포함) [1]\n- 수행 기..."
7,D_nano_4500,gpt-5-nano,4500,차세대 학사 정보시스템 구축에 대해 알려줘,8.27,3235,3072,1181,1152,0.00050,다음은 차세대 학사 정보시스템 구축에 대해 맥락 문서의 내용을 바탕으로 정리한 요약...
8,E_nano_8000,gpt-5-nano,8000,한영대학교 특성화 맞춤형 교육환경 구축 사업의 예산과 수행 기간은 어떻게 돼?,2.95,3459,3328,242,175,0.00012,"- 예산: 130,000,000원(부가가치세 포함) [1]\n- 수행 기간: 계약일..."
9,E_nano_8000,gpt-5-nano,8000,차세대 학사 정보시스템 구축에 대해 알려줘,7.71,3235,3072,1004,1167,0.00043,다음은 제공된 문서에 근거한 차세대 학사 정보시스템 구축의 주요 내용 요약입니다.\...


In [16]:
# 참고: 이 셀은 초기 탐색용(6개 config 후보 좁히기 목적)이며, success 기준이 "답변이 비어있지 않음"입니다.
# 정답 키워드 기준 최종 검증은 아래 "셀 3: 신뢰성 검증" 결과를 따릅니다.
N_REPEAT = 3

FOCUS_CONFIGS = {
    "B_nano_1500": {"model": "gpt-5-nano", "max_tokens": 1500},
    "A_nano_3000": {"model": "gpt-5-nano", "max_tokens": 3000},
    "D_nano_4500": {"model": "gpt-5-nano", "max_tokens": 4500},
    "E_nano_8000": {"model": "gpt-5-nano", "max_tokens": 8000},
    "F_mini_1500": {"model": "gpt-5-mini", "max_tokens": 1500},
    "C_mini_3000": {"model": "gpt-5-mini", "max_tokens": 3000},
}

Q2 = "차세대 학사 정보시스템 구축에 대해 알려줘"

repeat_results = []
for cfg_name, overrides in FOCUS_CONFIGS.items():
    tmp_path = make_temp_config(overrides)
    for trial in range(N_REPEAT):
        start = time.perf_counter()
        with get_openai_callback() as cb:
            response = run(Q2, config_path=tmp_path, profile="openai")
        elapsed = time.perf_counter() - start
        cost = cb.total_cost
        repeat_results.append({
            "config": cfg_name, "trial": trial + 1,
            "latency_sec": round(elapsed, 2),
            "completion_tokens": cb.completion_tokens,
            "answer_len": len(response["answer"]),
            "success": len(response["answer"]) > 0,
            "cost_usd": round(cost, 5),
        })

df_repeat = pd.DataFrame(repeat_results)
summary = df_repeat.groupby("config").agg(
    success_rate=("success", "mean"),
    avg_completion_tokens=("completion_tokens", "mean"),
    avg_latency_sec=("latency_sec", "mean"),
    avg_cost_usd=("cost_usd", "mean"),
).round(3)
summary

,success_rate,avg_completion_tokens,avg_latency_sec,avg_cost_usd
config,,,,
A_nano_3000,1.0,1030.000,8.763,0.000
B_nano_1500,1.0,861.000,6.803,0.000
C_mini_3000,1.0,846.333,11.143,0.002
D_nano_4500,1.0,1070.000,8.143,0.000
E_nano_8000,1.0,993.667,7.377,0.000
F_mini_1500,1.0,854.000,14.933,0.002


In [17]:
targets = [
    ("A_nano_3000", "한영대학교 특성화 맞춤형 교육환경 구축 사업의 예산과 수행 기간은 어떻게 돼?"),
    ("C_mini_3000", "한영대학교 특성화 맞춤형 교육환경 구축 사업의 예산과 수행 기간은 어떻게 돼?"),
    ("C_mini_3000", "차세대 학사 정보시스템 구축에 대해 알려줘"),
]

for cfg, q in targets:
    row = df[(df["config"] == cfg) & (df["question"] == q)].iloc[0]
    print(f"=== {cfg} | {q} ===")
    print(row["answer"])
    print()

=== A_nano_3000 | 한영대학교 특성화 맞춤형 교육환경 구축 사업의 예산과 수행 기간은 어떻게 돼? ===
- 예산: 130,000,000원(VAT 포함) [1]
- 수행 기간: 계약일로부터 3개월(안정화기간 1개월 포함) [1]

참고: 한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화 제안요청서 [1]

=== C_mini_3000 | 한영대학교 특성화 맞춤형 교육환경 구축 사업의 예산과 수행 기간은 어떻게 돼? ===
- 사업예산: 130,000,000원(부가가치세 포함)으로 책정되어 있음 [1].  
- 사업기간: 계약일로부터 3개월(안정화기간 1개월 포함)으로 명시되어 있음 [1].  

참고문서: 한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화 (한영대학) [1]

=== C_mini_3000 | 차세대 학사 정보시스템 구축에 대해 알려줘 ===
요청하신 “차세대 학사 정보시스템 구축”에 대해 제공된 문서들을 근거로 핵심 내용만 간결히 정리해 드립니다.

- 사업 목적 및 범위
  - 학사행정의 정보화 수준 향상 및 표준화된 통합 학사행정시스템 구축을 목표로 함[5][2].  
  - 서울·세종 등 복수 캠퍼스의 제도 차이를 반영하되 표준화된 시스템으로 구현하되, 캠퍼스별 조회/입력 등 처리는 구분 처리 가능하도록 구현해야 함[2].

- 업무프로세스 재설계(BPR)
  - 기존 문제(업무 이슈)를 재설계하고 정보화 방안 제시를 포함한 표준 업무프로세스 수립이 요구됨[1].  
  - 재설계 대상 주요 업무로는 졸업기준 및 정보화 수준 분석, 대학원 입시·학적 생성 시점에 따른 프로세스 개선, 장학금 관리 및 장학 신청·선발 전 과정의 정보화, 학적·성적·장학 신청 프로세스의 진행상태 알람 기능, 비학위과정(최고위과정 등)의 관리 프로세스 재설계 등이 포함됨[1].

- 장학관리 개선 요구
  - 장학금 코드 관리 개선(장학금 정보와 지급기준 정보 분리), 장학추천서비스 구현, 장학 신청→선

In [18]:
import sys
sys.path.insert(0, "..")  # 노트북이 notebook/ 폴더 안에 있다면
from api_main import run
from langchain_community.callbacks import get_openai_callback

with get_openai_callback() as cb:
    response = run("한영대학교 특성화 맞춤형 교육환경 구축 사업의 예산과 수행 기간은 어떻게 돼?", profile="openai")
print(vars(cb))

{'_lock': <unlocked _thread.lock object at 0x7b96c91e6100>, 'total_cost': 0.00023455, 'total_tokens': 3613, 'prompt_tokens': 3459, 'prompt_tokens_cached': 0, 'completion_tokens': 154, 'reasoning_tokens': 64, 'successful_requests': 1}


In [19]:
with get_openai_callback() as cb1:
    run("한영대학교 특성화 맞춤형 교육환경 구축 사업의 예산과 수행 기간은 어떻게 돼?", profile="openai")
print("1차:", vars(cb1))

with get_openai_callback() as cb2:
    run("한영대학교 특성화 맞춤형 교육환경 구축 사업의 예산과 수행 기간은 어떻게 돼?", profile="openai")
print("2차:", vars(cb2))

1차: {'_lock': <unlocked _thread.lock object at 0x7b968a4a6600>, 'total_cost': 0.00011119000000000001, 'total_tokens': 3679, 'prompt_tokens': 3459, 'prompt_tokens_cached': 3328, 'completion_tokens': 220, 'reasoning_tokens': 128, 'successful_requests': 1}
2차: {'_lock': <unlocked _thread.lock object at 0x7b968232ff00>, 'total_cost': 8.799e-05, 'total_tokens': 3621, 'prompt_tokens': 3459, 'prompt_tokens_cached': 3328, 'completion_tokens': 162, 'reasoning_tokens': 64, 'successful_requests': 1}


In [20]:
# --- 셀 1: CONFIGS 비교 (1회씩) ---
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
import time, yaml
from copy import deepcopy
from langchain_community.callbacks import get_openai_callback
from api_main import run, load_config

BASE_CONFIG_PATH = "config/default.yaml"

CONFIGS = {
    "A_nano_3000": {"model": "gpt-5-nano", "max_tokens": 3000},
    "B_nano_1500": {"model": "gpt-5-nano", "max_tokens": 1500},
    "C_mini_3000":  {"model": "gpt-5-mini", "max_tokens": 3000},
    "D_nano_4500":  {"model": "gpt-5-nano", "max_tokens": 4500},
}

# 공통질문(questions.jsonl) 중 baseline(Q01) + 실제로 빈 답변이 나왔던 계산 질문(Q02, Q07)
TARGET_QUESTION_IDS = ["Q01", "Q02", "Q07"]
QUESTIONS = [QUESTIONS_BY_ID[qid]["question"] for qid in TARGET_QUESTION_IDS]
KEYWORDS_BY_QUESTION = {QUESTIONS_BY_ID[qid]["question"]: QUESTIONS_BY_ID[qid]["keywords"] for qid in TARGET_QUESTION_IDS}

def make_temp_config(overrides: dict) -> str:
    cfg = deepcopy(load_config(BASE_CONFIG_PATH))
    cfg["generation"].update(overrides)
    tmp_path = f"config/_tmp_{overrides['model']}_{overrides['max_tokens']}.yaml"
    with open(tmp_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(cfg, f, allow_unicode=True)
    return tmp_path

results = []
for cfg_name, overrides in CONFIGS.items():
    tmp_path = make_temp_config(overrides)
    for q in QUESTIONS:
        start = time.perf_counter()
        with get_openai_callback() as cb:
            response = run(q, config_path=tmp_path, profile="openai")
        elapsed = time.perf_counter() - start
        results.append({
            "config": cfg_name, "model": overrides["model"], "max_tokens": overrides["max_tokens"],
            "question": q, "latency_sec": round(elapsed, 2),
            "prompt_tokens": cb.prompt_tokens, "prompt_tokens_cached": cb.prompt_tokens_cached,
            "completion_tokens": cb.completion_tokens,
            "answer_len": len(response["answer"]),
            "cost_usd": round(cb.total_cost, 5), "answer": response["answer"],
        })

import pandas as pd
df = pd.DataFrame(results)
df

,config,model,max_tokens,question,latency_sec,prompt_tokens,prompt_tokens_cached,completion_tokens,answer_len,cost_usd,answer
0,A_nano_3000,gpt-5-nano,3000,한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화 사업의 예...,3.44,3406,0,228,146,0.00026,"- 예산: 130,000,000원(VAT 포함) [1][2]\n- 수행 기간: 계약..."
1,A_nano_3000,gpt-5-nano,3000,"한국연구재단 UICC 기능개선 사업의 전체 사업비와 SW 개발비를 알려주고, SW ...",4.07,3332,0,456,216,0.00035,"- 전체 사업비: 129,300,000원 (부가세 포함) [4]\n- SW 개발비:..."
2,A_nano_3000,gpt-5-nano,3000,한영대학교 트랙운영 사업과 한국연구재단 UICC 사업의 예산과 수행 기간을 비교하고...,5.31,3553,0,608,587,0.00042,- 요약\n - 한국연구재단 UICC(대학 산학협력활동 실태조사 시스템 기능개선)...
3,B_nano_1500,gpt-5-nano,1500,한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화 사업의 예...,2.69,3406,3328,214,122,0.00011,"- 예산: 130,000,000원(VAT 포함) 범위 내\n- 수행 기간: 계약일로..."
4,B_nano_1500,gpt-5-nano,1500,"한국연구재단 UICC 기능개선 사업의 전체 사업비와 SW 개발비를 알려주고, SW ...",4.25,3332,3200,526,220,0.00023,"- 전체 사업비: 129,300,000원(부가세 포함)으로 제시됩니다 [2][4]...."
5,B_nano_1500,gpt-5-nano,1500,한영대학교 트랙운영 사업과 한국연구재단 UICC 사업의 예산과 수행 기간을 비교하고...,5.81,3553,3456,654,453,0.00028,- 한영대학교 트랙운영 학사정보시스템 고도화\n - 예산: 문서에서 명시되어 있지...
6,C_mini_3000,gpt-5-mini,3000,한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화 사업의 예...,3.19,3406,3328,159,131,0.00042,"- 사업예산: 130,000,000원(부가가치세 포함) [1] \n- 사업기간: ..."
7,C_mini_3000,gpt-5-mini,3000,"한국연구재단 UICC 기능개선 사업의 전체 사업비와 SW 개발비를 알려주고, SW ...",5.88,3332,3200,391,203,0.00090,"- 전체 사업비: 129,300,000원(부가세 포함) [4]. \n- SW 개발..."
8,C_mini_3000,gpt-5-mini,3000,한영대학교 트랙운영 사업과 한국연구재단 UICC 사업의 예산과 수행 기간을 비교하고...,6.60,3553,0,450,423,0.00179,- 한국연구재단의 2024년 대학 산학협력활동 실태조사 시스템(UICC) 기능개선 ...
9,D_nano_4500,gpt-5-nano,4500,한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화 사업의 예...,4.23,3406,3328,420,142,0.00019,"- 예산: 130,000,000원(VAT 포함) [1]\n- 수행 기간: 계약일로부..."


In [21]:
# --- 셀 2: NEW_CONFIGS 비교 (1회씩) ---
NEW_CONFIGS = {
    "E_nano_8000": {"model": "gpt-5-nano", "max_tokens": 8000},
    "F_mini_1500": {"model": "gpt-5-mini", "max_tokens": 1500},
}

new_results = []
for cfg_name, overrides in NEW_CONFIGS.items():
    tmp_path = make_temp_config(overrides)
    for q in QUESTIONS:
        start = time.perf_counter()
        with get_openai_callback() as cb:
            response = run(q, config_path=tmp_path, profile="openai")
        elapsed = time.perf_counter() - start
        new_results.append({
            "config": cfg_name, "model": overrides["model"], "max_tokens": overrides["max_tokens"],
            "question": q, "latency_sec": round(elapsed, 2),
            "prompt_tokens": cb.prompt_tokens, "prompt_tokens_cached": cb.prompt_tokens_cached,
            "completion_tokens": cb.completion_tokens,
            "answer_len": len(response["answer"]),
            "cost_usd": round(cb.total_cost, 5), "answer": response["answer"],
        })

df2 = pd.DataFrame(new_results)
df_all = pd.concat([df, df2], ignore_index=True)
df_all

,config,model,max_tokens,question,latency_sec,prompt_tokens,prompt_tokens_cached,completion_tokens,answer_len,cost_usd,answer
0,A_nano_3000,gpt-5-nano,3000,한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화 사업의 예...,3.44,3406,0,228,146,0.00026,"- 예산: 130,000,000원(VAT 포함) [1][2]\n- 수행 기간: 계약..."
1,A_nano_3000,gpt-5-nano,3000,"한국연구재단 UICC 기능개선 사업의 전체 사업비와 SW 개발비를 알려주고, SW ...",4.07,3332,0,456,216,0.00035,"- 전체 사업비: 129,300,000원 (부가세 포함) [4]\n- SW 개발비:..."
2,A_nano_3000,gpt-5-nano,3000,한영대학교 트랙운영 사업과 한국연구재단 UICC 사업의 예산과 수행 기간을 비교하고...,5.31,3553,0,608,587,0.00042,- 요약\n - 한국연구재단 UICC(대학 산학협력활동 실태조사 시스템 기능개선)...
3,B_nano_1500,gpt-5-nano,1500,한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화 사업의 예...,2.69,3406,3328,214,122,0.00011,"- 예산: 130,000,000원(VAT 포함) 범위 내\n- 수행 기간: 계약일로..."
4,B_nano_1500,gpt-5-nano,1500,"한국연구재단 UICC 기능개선 사업의 전체 사업비와 SW 개발비를 알려주고, SW ...",4.25,3332,3200,526,220,0.00023,"- 전체 사업비: 129,300,000원(부가세 포함)으로 제시됩니다 [2][4]...."
5,B_nano_1500,gpt-5-nano,1500,한영대학교 트랙운영 사업과 한국연구재단 UICC 사업의 예산과 수행 기간을 비교하고...,5.81,3553,3456,654,453,0.00028,- 한영대학교 트랙운영 학사정보시스템 고도화\n - 예산: 문서에서 명시되어 있지...
6,C_mini_3000,gpt-5-mini,3000,한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화 사업의 예...,3.19,3406,3328,159,131,0.00042,"- 사업예산: 130,000,000원(부가가치세 포함) [1] \n- 사업기간: ..."
7,C_mini_3000,gpt-5-mini,3000,"한국연구재단 UICC 기능개선 사업의 전체 사업비와 SW 개발비를 알려주고, SW ...",5.88,3332,3200,391,203,0.00090,"- 전체 사업비: 129,300,000원(부가세 포함) [4]. \n- SW 개발..."
8,C_mini_3000,gpt-5-mini,3000,한영대학교 트랙운영 사업과 한국연구재단 UICC 사업의 예산과 수행 기간을 비교하고...,6.60,3553,0,450,423,0.00179,- 한국연구재단의 2024년 대학 산학협력활동 실태조사 시스템(UICC) 기능개선 ...
9,D_nano_4500,gpt-5-nano,4500,한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화 사업의 예...,4.23,3406,3328,420,142,0.00019,"- 예산: 130,000,000원(VAT 포함) [1]\n- 수행 기간: 계약일로부..."


In [22]:
# --- 셀 3: 신뢰성 검증 (후보 3개 설정 × 3개 질문 × 3회 반복) ---
N_REPEAT = 3
FOCUS_CONFIGS = {
    "D_nano_4500": {"model": "gpt-5-nano", "max_tokens": 4500},
    "E_nano_8000": {"model": "gpt-5-nano", "max_tokens": 8000},
    "C_mini_3000": {"model": "gpt-5-mini", "max_tokens": 3000},
}

repeat_results = []
for cfg_name, overrides in FOCUS_CONFIGS.items():
    tmp_path = make_temp_config(overrides)
    for q in QUESTIONS:
        for trial in range(N_REPEAT):
            start = time.perf_counter()
            with get_openai_callback() as cb:
                response = run(q, config_path=tmp_path, profile="openai")
            elapsed = time.perf_counter() - start
            answer = response["answer"]
            keywords = KEYWORDS_BY_QUESTION[q]
            keyword_hit = sum(1 for kw in keywords if kw in answer)
            repeat_results.append({
                "config": cfg_name, "question": q, "trial": trial + 1,
                "latency_sec": round(elapsed, 2),
                "prompt_tokens_cached": cb.prompt_tokens_cached,
                "completion_tokens": cb.completion_tokens,
                "answer_len": len(answer),
                "keyword_hit": keyword_hit,
                "keyword_total": len(keywords),
                "success": keyword_hit == len(keywords),  # 정답 키워드 전부 포함해야 성공
                "cost_usd": round(cb.total_cost, 5),
            })

df_repeat = pd.DataFrame(repeat_results)
summary = df_repeat.groupby(["config", "question"]).agg(
    success_rate=("success", "mean"),
    avg_completion_tokens=("completion_tokens", "mean"),
    avg_latency_sec=("latency_sec", "mean"),
    avg_cost_usd=("cost_usd", "mean"),
).round(4)
summary

success_rate  \
config      question                                                           
C_mini_3000 한국연구재단 UICC 기능개선 사업의 전체 사업비와 SW 개발비를 알려주고, SW 개...           1.0   
            한영대학교 트랙운영 사업과 한국연구재단 UICC 사업의 예산과 수행 기간을 비교하고 ...           0.0   
            한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화 사업의 예산...           1.0   
D_nano_4500 한국연구재단 UICC 기능개선 사업의 전체 사업비와 SW 개발비를 알려주고, SW 개...           1.0   
            한영대학교 트랙운영 사업과 한국연구재단 UICC 사업의 예산과 수행 기간을 비교하고 ...           0.0   
            한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화 사업의 예산...           1.0   
E_nano_8000 한국연구재단 UICC 기능개선 사업의 전체 사업비와 SW 개발비를 알려주고, SW 개...           1.0   
            한영대학교 트랙운영 사업과 한국연구재단 UICC 사업의 예산과 수행 기간을 비교하고 ...           0.0   
            한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화 사업의 예산...           1.0   

                                                                avg_completion_tokens  \
config      question                                                                    
C_mini_3000 한국연구재단 UICC 기능개선 사업의 전체 사업비와 SW 개발비를 알려주고, SW 개...               317.6667   
            한영대학교 트랙운영 사업과 한국연구재단 UICC 사업의 예산과 수행 기간을 비교하고 ...               458.0000   
            한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화 사업의 예산...               186.6667   
D_nano_4500 한국연구재단 UICC 기능개선 사업의 전체 사업비와 SW 개발비를 알려주고, SW 개...               449.3333   
            한영대학교 트랙운영 사업과 한국연구재단 UICC 사업의 예산과 수행 기간을 비교하고 ...               660.0000   
            한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화 사업의 예산...               225.6667   
E_nano_8000 한국연구재단 UICC 기능개선 사업의 전체 사업비와 SW 개발비를 알려주고, SW 개...               476.3333   
            한영대학교 트랙운영 사업과 한국연구재단 UICC 사업의 예산과 수행 기간을 비교하고 ...               632.0000   
            한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화 사업의 예산...               222.3333   

                                                                avg_latency_sec  \
config      question                                                              
C_mini_3000 한국연구재단 UICC 기능개선 사업의 전체 사업비와 SW 개발비를 알려주고, SW 개...           5.5300   
            한영대학교 트랙운영 사업과 한국연구재단 UICC 사업의 예산과 수행 기간을 비교하고 ...           6.6800   
            한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화 사업의 예산...           3.6100   
D_nano_4500 한국연구재단 UICC 기능개선 사업의 전체 사업비와 SW 개발비를 알려주고, SW 개...           5.4467   
            한영대학교 트랙운영 사업과 한국연구재단 UICC 사업의 예산과 수행 기간을 비교하고 ...           6.7033   
            한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화 사업의 예산...           3.7833   
E_nano_8000 한국연구재단 UICC 기능개선 사업의 전체 사업비와 SW 개발비를 알려주고, SW 개...           5.5567   
            한영대학교 트랙운영 사업과 한국연구재단 UICC 사업의 예산과 수행 기간을 비교하고 ...           5.1333   
            한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화 사업의 예산...           2.7367   

                                                                avg_cost_usd  
config      question                                                          
C_mini_3000 한국연구재단 UICC 기능개선 사업의 전체 사업비와 SW 개발비를 알려주고, SW 개...        0.0007  
            한영대학교 트랙운영 사업과 한국연구재단 UICC 사업의 예산과 수행 기간을 비교하고 ...        0.0010  
            한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화 사업의 예산...        0.0005  
D_nano_4500 한국연구재단 UICC 기능개선 사업의 전체 사업비와 SW 개발비를 알려주고, SW 개...        0.0002  
            한영대학교 트랙운영 사업과 한국연구재단 UICC 사업의 예산과 수행 기간을 비교하고 ...        0.0003  
            한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화 사업의 예산...        0.0002  
E_nano_8000 한국연구재단 UICC 기능개선 사업의 전체 사업비와 SW 개발비를 알려주고, SW 개...        0.0003  
            한영대학교 트랙운영 사업과 한국연구재단 UICC 사업의 예산과 수행 기간을 비교하고 ...        0.0003  
            한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화 사업의 예산...        0.0001

In [29]:
import time
import pandas as pd
from api_main import load_config, load_chunks
from src.retriever_factory import create_retriever
from src.rag_engine import build_context, condense_question, SYSTEM_RULE
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.callbacks import get_openai_callback

REASONING_QUESTION_IDS = ["Q01", "Q02", "Q05", "Q07"]
REASONING_TEST_QUESTIONS = [
    (qid, QUESTIONS_BY_ID[qid]["question"], QUESTIONS_BY_ID[qid]["keywords"])
    for qid in REASONING_QUESTION_IDS
]

config = load_config()
chunks = load_chunks(config["paths"]["chunks"])
retriever = create_retriever(chunks, config["retrieval"], "openai")
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_RULE + "\n\nContext:\n{context}"),
    MessagesPlaceholder("history", optional=True),
    ("human", "{question}")
])

N_REPEAT = 3
reasoning_results = []
for effort in ["low", "medium"]:
    llm = ChatOpenAI(model=config["generation"]["model"], max_tokens=config["generation"]["max_tokens"], reasoning_effort=effort)
    for qid, q, keywords in REASONING_TEST_QUESTIONS:
        search_q = condense_question(q, None, config)
        results = retriever.search(search_q, top_k=config["retrieval"]["top_k"])
        context = build_context(results)
        messages = prompt.format_messages(context=context, question=q, history=[])
        for trial in range(N_REPEAT):
            start = time.perf_counter()
            with get_openai_callback() as cb:
                ai_msg = llm.invoke(messages)
            elapsed = time.perf_counter() - start
            answer = ai_msg.content
            keyword_hit = sum(1 for kw in keywords if kw in answer)
            reasoning_results.append({
                "effort": effort, "question_id": qid, "trial": trial + 1,
                "latency_sec": round(elapsed, 2),
                "completion_tokens": cb.completion_tokens,
                "hit_ceiling": cb.completion_tokens >= config["generation"]["max_tokens"],
                "answer_len": len(answer),
                "keyword_hit": keyword_hit,
                "keyword_total": len(keywords),
                "cost_usd": round(cb.total_cost, 5),
                "answer_preview": answer[:80],
            })
df_reasoning = pd.DataFrame(reasoning_results)
df_reasoning

,effort,question_id,trial,latency_sec,completion_tokens,hit_ceiling,answer_len,keyword_hit,keyword_total,cost_usd,answer_preview
0,low,Q01,1,3.19,278,False,123,3,3,0.00028,"- 예산: 130,000,000원(VAT 포함) [1]\n- 수행 기간: 계약일로부..."
1,low,Q01,2,3.42,358,False,149,3,3,0.00016,"- 예산: 130,000,000원(VAT 포함) [1][2]\n- 수행 기간: 계약..."
2,low,Q01,3,3.28,218,False,125,3,3,0.00011,"- 예산: 130,000,000원(부가가치세 포함) 범위 내\n- 수행 기간: 계약..."
3,low,Q02,1,4.38,573,False,285,2,2,0.00040,"- 전체 사업비: 129,300,000원(부가세 포함) [4][2]\n- SW 개발..."
4,low,Q02,2,3.99,378,False,183,2,2,0.00032,"- 전체 사업비: 129,300,000원(부가세 포함) [4]\n- SW 개발비: ..."
5,low,Q02,3,4.48,585,False,208,2,2,0.00026,"- 전체 사업비: 129,300,000원(부가세 포함) [4]\n- SW 개발비: ..."
6,low,Q05,1,5.67,502,False,192,0,6,0.00037,- 제공된 맥락 문서에는 서울시립대학교(서울시립대) 사업에 대한 구체적인 부서 배정...
7,low,Q05,2,5.77,719,False,666,0,6,0.00046,다음의 요청은 제공된 문서들에서 직접적으로 확인할 수 없습니다.\n\n- 요청하신 ...
8,low,Q05,3,4.40,430,False,498,0,6,0.00034,"죄송하지만 제공된 문서들에는 서울시립대학교의 대입전형 자료, 학적 정보, 학생 활동..."
9,low,Q07,1,4.65,591,False,456,2,4,0.00041,- 한영대학교 트랙운영 사업\n - 제공된 문서에서 예산 및 수행 기간에 대한 구...


In [30]:
summary_reasoning = df_reasoning.groupby(["effort", "question_id"]).agg(
    avg_keyword_hit=("keyword_hit", "mean"),
    keyword_total=("keyword_total", "first"),
    avg_completion_tokens=("completion_tokens", "mean"),
    avg_latency_sec=("latency_sec", "mean"),
    avg_cost_usd=("cost_usd", "mean"),
    hit_ceiling_rate=("hit_ceiling", "mean"),
).round(4)
summary_reasoning

avg_keyword_hit  keyword_total  avg_completion_tokens  \
effort question_id                                                          
low    Q01                   3.0000              3               284.6667   
       Q02                   2.0000              2               512.0000   
       Q05                   0.0000              6               550.3333   
       Q07                   2.0000              4               615.0000   
medium Q01                   3.0000              3              1370.6667   
       Q02                   1.3333              2              4052.6667   
       Q05                   0.0000              6              3962.3333   
       Q07                   2.0000              4              2402.3333   

                    avg_latency_sec  avg_cost_usd  hit_ceiling_rate  
effort question_id                                                   
low    Q01                   3.2967        0.0002            0.0000  
       Q02                   4.2833        0.0003            0.0000  
       Q05                   5.2800        0.0004            0.0000  
       Q07                   4.7433        0.0004            0.0000  
medium Q01                  16.8900        0.0007            0.0000  
       Q02                  25.9667        0.0017            0.3333  
       Q05                  27.8667        0.0017            0.3333  
       Q07                  14.1900        0.0010            0.0000

In [25]:
# Q05 리트리버가 실제로 뭘 가져왔는지 직접 확인
q05 = QUESTIONS_BY_ID["Q05"]["question"]
search_q = condense_question(q05, None, config)
retrieved = retriever.search(search_q, top_k=config["retrieval"]["top_k"])

print(f"검색된 chunk 수: {len(retrieved)}\n")
for i, r in enumerate(retrieved, 1):
    has_keywords = [kw for kw in QUESTIONS_BY_ID["Q05"]["keywords"] if kw in r.text]
    print(f"--- [{i}] doc_id={r.doc_id} chunk_id={r.chunk_id} score={r.score:.4f} ---")
    print(f"포함된 키워드: {has_keywords if has_keywords else '없음'}")
    print(r.text[:300])
    print()

검색된 chunk 수: 5

--- [1] doc_id=doc_077 chunk_id=doc_077_chunk_0087 score=0.4997 ---
포함된 키워드: 없음
 남서울대학교로 부터 업무수행을 위하여 명시적으로 접근을 허가받은 시설과 정보만을 이용하겠습니다. 또한, 남서울대학교의 업무를 위하여 제공한 인터넷, E-mail, fax, 전화 등에 대해서는 비밀보호를 위하여 남서울대학교의 일정한 제한이나 통제를 할 수 있음을 인정하고, 이에 동의합니다. 3. 본인은 남서울대학교의 사전 승인 없이는 남서울대학교의 비밀 정보를 외부로 반출하지 않겠습니다. 4. 본인은 업무가 종결되거나, 남서울대학교의 요청이 있는 경우 남서울대학교에 제공한 모든 자료와 자산을 남서울대학교에 즉시 반납하겠습니다. 5.

--- [2] doc_id=doc_008 chunk_id=doc_008_chunk_0074 score=0.4675 ---
포함된 키워드: 없음
 • 상벌사항 • 학생자치기구(동아리) 활동 내역 • 교환교류 프로그램 참여 내역 • 경력/학력내역 등 - 학생 전체 정보는 포털을 통해 학생 본인은 모두 조회할 수 있으며, 본부부서(학사팀, 교무학사팀, 대학원행정팀), 학사담당(대학행정실, 학과, 특수/전문 대학원행정실 등), 교원은 각 업무에 필요한 정보를 조회 - 사진을 등록하고 변경 이력을 관리할 수 있도록 기능 구현 - 계좌(은행, 예금주, 계좌번호) 정보 변경 시 인증 기능 구현 - 학생신상 변경 이력을 관리함(변경자, 변경일자, 변경사유, 변경항목 등) - 학부와 대학

--- [3] doc_id=doc_045 chunk_id=doc_045_chunk_0004 score=0.4566 ---
포함된 키워드: ['교무처']
템과의 연동  외부기관 연계 - 한국연구재단, 한국장학재단, 병무청 등 외부기관과의 정보 연계 ※ 구축 및 도입 관련 상세 내용은 제안요구사항의 각 요구기능을 참고 바람. Ⅱ 대학현황 및 문제점 1 일반 현황 가. 조직도-광주 나. 조직도-

In [28]:
llm = ChatOpenAI(model=config["generation"]["model"], max_tokens=config["generation"]["max_tokens"], reasoning_effort=config["generation"]["reasoning_effort"])

COVERAGE_QUESTION_IDS = ["Q01","Q02","Q03","Q04","Q05","Q06","Q07","Q08","Q09","Q10","Q12","Q13"]  # Q11은 이미지 근거 필요, 이슈#47 전까지 제외
ALL_13_QUESTIONS = [QUESTIONS_BY_ID[qid]["question"] for qid in COVERAGE_QUESTION_IDS]

coverage_low_results = []
for q in ALL_13_QUESTIONS:  # Q11 제외, 아까 쓴 12개 리스트 재사용
    search_q = condense_question(q, None, config)
    results = retriever.search(search_q, top_k=config["retrieval"]["top_k"])
    context = build_context(results)
    messages = prompt.format_messages(context=context, question=q, history=[])

    start = time.perf_counter()
    with get_openai_callback() as cb:
        ai_msg = llm.invoke(messages)
    elapsed = time.perf_counter() - start

    coverage_low_results.append({
        "question": q[:30], "latency_sec": round(elapsed, 2),
        "completion_tokens": cb.completion_tokens,
        "hit_ceiling": cb.completion_tokens >= config["generation"]["max_tokens"],
        "answer_len": len(ai_msg.content),
        "cost_usd": round(cb.total_cost, 5),
    })

df_coverage_low = pd.DataFrame(coverage_low_results)
df_coverage_low

,question,latency_sec,completion_tokens,hit_ceiling,answer_len,cost_usd
0,한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학,2.85,279,False,124,0.00028
1,한국연구재단 UICC 기능개선 사업의 전체 사업비와 S,3.29,407,False,238,0.00033
2,한국연구재단 UICC 기능개선 사업에서 데이터 입력 기,3.99,458,False,367,0.00034
3,스포츠윤리센터 LMS 기능개선 사업의 교육 미디어 플레,4.42,396,False,249,0.00033
4,"서울시립대학교 사업에서 대입전형 자료, 학적 정보, 학",3.98,519,False,539,0.00038
5,서울시립대학교의 대학생활만족도 정보는 어느 부서가 담당,3.20,360,False,159,0.00031
6,한영대학교 트랙운영 사업과 한국연구재단 UICC 사업의,4.03,591,False,347,0.00041
7,한국연구재단 UICC 기능개선 사업에서 블록체인 도입을,2.12,196,False,210,0.00024
8,스포츠윤리센터 LMS 기능개선 사업의 미디어 플레이어,2.05,232,False,184,0.00026
9,그 요구사항 중 자막 기능도 포함되어 있나?,4.35,524,False,324,0.00037


In [31]:
high_results = []
llm_high = ChatOpenAI(model=config["generation"]["model"], max_tokens=config["generation"]["max_tokens"], reasoning_effort="high")
for qid, q, keywords in REASONING_TEST_QUESTIONS:
    search_q = condense_question(q, None, config)
    results = retriever.search(search_q, top_k=config["retrieval"]["top_k"])
    context = build_context(results)
    messages = prompt.format_messages(context=context, question=q, history=[])
    for trial in range(N_REPEAT):
        start = time.perf_counter()
        with get_openai_callback() as cb:
            ai_msg = llm_high.invoke(messages)
        elapsed = time.perf_counter() - start
        answer = ai_msg.content
        keyword_hit = sum(1 for kw in keywords if kw in answer)
        high_results.append({
            "effort": "high", "question_id": qid, "trial": trial + 1,
            "latency_sec": round(elapsed, 2),
            "completion_tokens": cb.completion_tokens,
            "hit_ceiling": cb.completion_tokens >= config["generation"]["max_tokens"],
            "answer_len": len(answer),
            "keyword_hit": keyword_hit,
            "keyword_total": len(keywords),
            "cost_usd": round(cb.total_cost, 5),
            "answer_preview": answer[:80],
        })
df_high = pd.DataFrame(high_results)
df_reasoning_all = pd.concat([df_reasoning, df_high], ignore_index=True)

summary_reasoning_all = df_reasoning_all.groupby(["effort", "question_id"]).agg(
    avg_keyword_hit=("keyword_hit", "mean"),
    keyword_total=("keyword_total", "first"),
    avg_completion_tokens=("completion_tokens", "mean"),
    avg_latency_sec=("latency_sec", "mean"),
    avg_cost_usd=("cost_usd", "mean"),
    hit_ceiling_rate=("hit_ceiling", "mean"),
).round(4)
summary_reasoning_all

avg_keyword_hit  keyword_total  avg_completion_tokens  \
effort question_id                                                          
high   Q01                   2.0000              3              3223.3333   
       Q02                   0.0000              2              4500.0000   
       Q05                   0.0000              6              4500.0000   
       Q07                   0.0000              4              4500.0000   
low    Q01                   3.0000              3               284.6667   
       Q02                   2.0000              2               512.0000   
       Q05                   0.0000              6               550.3333   
       Q07                   2.0000              4               615.0000   
medium Q01                   3.0000              3              1370.6667   
       Q02                   1.3333              2              4052.6667   
       Q05                   0.0000              6              3962.3333   
       Q07                   2.0000              4              2402.3333   

                    avg_latency_sec  avg_cost_usd  hit_ceiling_rate  
effort question_id                                                   
high   Q01                  19.2800        0.0014            0.3333  
       Q02                  36.4267        0.0019            1.0000  
       Q05                  25.6267        0.0019            1.0000  
       Q07                  22.8300        0.0019            1.0000  
low    Q01                   3.2967        0.0002            0.0000  
       Q02                   4.2833        0.0003            0.0000  
       Q05                   5.2800        0.0004            0.0000  
       Q07                   4.7433        0.0004            0.0000  
medium Q01                  16.8900        0.0007            0.0000  
       Q02                  25.9667        0.0017            0.3333  
       Q05                  27.8667        0.0017            0.3333  
       Q07                  14.1900        0.0010            0.0000

In [5]:
import yaml
from pathlib import Path
from src.parser_chunker import load_chunks_jsonl, demo_chunks
from src.retriever_factory import create_retriever
from src.rag_engine import generate_answer, condense_question

with open("config/default.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

chunks_path = Path(config["paths"]["chunks"])
chunks = load_chunks_jsonl(chunks_path) if chunks_path.exists() else demo_chunks()
retriever = create_retriever(chunks, config["retrieval"], "openai")  # #30 실험과 동일 profile

def ask(question, history=None):
    history = history or []
    search_question = condense_question(question, history, config)
    results = retriever.search(search_question, top_k=config["retrieval"]["top_k"], filters=None)
    response = generate_answer(question, results, config, history=history)
    print(f"[질문] {question}")
    print(f"[답변]\n{response['answer']}\n")
    print("-" * 60)
    return response

In [7]:
ask(QUESTIONS_BY_ID["Q01"]["question"], [])   # 계산 무관 — 회귀 확인
ask(QUESTIONS_BY_ID["Q02"]["question"], [])   # 계산 필요 — tool calling 확인
_ = ask(QUESTIONS_BY_ID["Q07"]["question"], [])   # 다문서 비교·계산 질문 확인

[질문] 한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화 사업의 예산과 수행 기간을 알려줘.
[답변]
- 예산: 130,000,000원(VAT 포함) [1]
- 사업기간: 계약일로부터 3개월(안정화기간 1개월 포함) [1]

참고한 문서: 한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화 [1]

------------------------------------------------------------
[질문] 한국연구재단 UICC 기능개선 사업의 전체 사업비와 SW 개발비를 알려주고, SW 개발비 비율을 계산해줘.
[답변]
- 제공된 문서상으로는 UICC 기능개선 사업의 전체 사업비가 명시되어 있지 않아 총액을 확인해 드릴 수 없습니다. 또한 SW 개발비는 92,300,000원 이내로 제시되어 있습니다. 따라서 SW 개발비의 비율(= SW 개발비 / 전체 사업비)은 현재 근거 자료만으로는 산출할 수 없습니다. [2]

참고한 문서 출처: 한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선.hwp [2]

------------------------------------------------------------
[질문] 한영대학교 트랙운영 사업과 한국연구재단 UICC 사업의 예산과 수행 기간을 비교하고 사업비 차이도 계산해줘.
[답변]
- 한국연구재단 UICC 사업
  - 예산: 129,300,000원(부가세 포함) [1]
  - 사업기간: 계약일로부터 210일 [1]

- 한영대학교 트랙운영 사업
  - 제공된 문서에서 예산 및 수행 기간에 대한 구체적 수치를 확인할 수 없습니다. 관련 항목은 문서에 명시되어 있지 않거나 누락된 상태로 보입니다. 따라서 동일한 기준으로 비교하기 어렵습니다. 근거 자료에서 예산 및 기간 정보를 확인할 수 없어 차이를 계산할 수 없습니다 [4].

- 결론
  - 현재 제시된 자료에 따르면 UICC 사업의 예산과 기간은 확인되나, 한영대학교 트랙운영 사업

In [3]:
from src.rag_engine import calculate

# 정상 케이스
print(calculate.invoke({"expression": "92300000 / 129300000 * 100"}))  # 71.38... 나와야 함
print(calculate.invoke({"expression": "(500-320)/320*100"}))            # 56.25
print(calculate.invoke({"expression": "-5 + 3 * 2"}))                   # 1

# 공격/오류 케이스 — 전부 "계산 오류"로 거부돼야 함
print(calculate.invoke({"expression": "__import__('os').system('ls')"}))
print(calculate.invoke({"expression": "open('x')"}))
print(calculate.invoke({"expression": "1/0"}))

71.38437741686002
56.25
1
계산 오류: 숫자와 사칙연산(+ - * / 괄호)만 사용할 수 있습니다
계산 오류: 숫자와 사칙연산(+ - * / 괄호)만 사용할 수 있습니다
계산 오류: 0으로 나눌 수 없습니다


In [4]:
# 검색 레이어 이슈(청크 잘림/score=0)를 우회한 Generation 격리 검증
# 완전한 숫자가 포함된 doc_002 청크를 직접 구성해 tool calling 성공 경로를 확인한다
from src.retriever import SearchResult
from src.rag_engine import generate_answer

dummy_results = [SearchResult(
    chunk_id="doc_002_chunk_0001",
    doc_id="doc_002",
    text=("4. 사업기간 및 사업비 □ 사업기간 : 계약일로부터 210일 "
          "□ 사업금액 : 129,300,000원(부가세 포함) ㅇ 예산 세부내역 구분 금액 비고 "
          "SW 개발비 92,300,000원 이내 · 실태조사 시스템 기능 개선 "
          "운영지원 및 데이터 점검 37,000,000원 이내 · 실태조사 시스템 운영지원"),
    metadata={"title": "2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선",
              "agency": "한국연구재단",
              "file_name": "한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선.hwp"},
    score=0.9,
)]

response = generate_answer(QUESTIONS_BY_ID["Q02"]["question"], dummy_results, config)
print(response["answer"])

- 전체 사업비: 129,300,000원(부가세 포함) [1]
- SW 개발비: 92,300,000원 이내 [1]
- SW 개발비 비율: 약 71.38% (92,300,000 ÷ 129,300,000 × 100) [1]

참고한 문서: 한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선.hwp [1]
